# Compact Convolutional Autoencoder for MNIST

Compact convolutional autoencoder for MNIST (28x28x1 grayscale digits,
10 classes, but labels are not used for training). Every encoder/decoder
operation is a named layer (no `nn.Sequential`), and the forward pass
prints the `[B, C, H, W]` (or `[B, D]` in the bottleneck) shape after
each individual operation.

**Architecture** (`ConvAutoencoder`, defined in
`code/mnist_conv_autoencoder.py`), shape trace for `latent_dim=32`:

```
input                          [B,  1, 28, 28]
--- encoder ---
conv1 3x3 s2 p1 (1->16)     -> [B, 16, 14, 14]
relu1                       -> [B, 16, 14, 14]
conv2 3x3 s2 p1 (16->32)    -> [B, 32,  7,  7]
relu2                       -> [B, 32,  7,  7]
conv3 3x3 s2 p1 (32->64)    -> [B, 64,  4,  4]
relu3                       -> [B, 64,  4,  4]
flatten                     -> [B, 1024]
fc_enc (1024->32)           -> [B,   32]          (latent z)
--- decoder ---
fc_dec (32->1024)           -> [B, 1024]
relu_dec                    -> [B, 1024]
unflatten                   -> [B, 64,  4,  4]
deconv1 3x3 s2 p1 op0 (64->32) -> [B, 32,  7,  7]
relu_d1                        -> [B, 32,  7,  7]
deconv2 3x3 s2 p1 op1 (32->16) -> [B, 16, 14, 14]
relu_d2                        -> [B, 16, 14, 14]
deconv3 3x3 s2 p1 op1 (16-> 1) -> [B,  1, 28, 28]
sigmoid                        -> [B,  1, 28, 28]   (x_hat in [0,1])
```

The model is trained to minimize pixel-wise reconstruction MSE,
`L(x) = mean((x - decoder(encoder(x)))^2)`, with no label supervision.

Core code lives in `code/mnist_conv_autoencoder.py` (version-controlled);
this notebook is a thin orchestration layer: GPU check, repo pull, build
model, print shape trace + parameter count, download real MNIST (cached
in Google Drive so it is downloaded only once), train, plot the loss
curve, and show original-vs-reconstruction qualitative + quantitative
results (per-image MSE) as the required results-visualization step.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

## 2. Clone/pull repository

Model code lives in `code/mnist_conv_autoencoder.py`, not duplicated in
this notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))

## 3. Build model, print shape trace and parameter count

One verbose forward pass on a small synthetic batch, purely to print
the `[B,C,H,W]` / `[B,D]` shape after every encoder/decoder op. Not
part of training.

In [ ]:
from mnist_conv_autoencoder import ConvAutoencoder, count_trainable_parameters

torch.manual_seed(0)

LATENT_DIM = 32

model = ConvAutoencoder(latent_dim=LATENT_DIM).to(device)
print(f"trainable parameters: {count_trainable_parameters(model):,}")

sample_batch = torch.rand(4, 1, 28, 28, device=device)  # [B,1,28,28], synthetic, in [0,1]
_ = model(sample_batch, verbose=True)

## 4. Download real MNIST (train/test split), cached in Drive

`torchvision.datasets.MNIST(train=True)` / `train=False` gives the
dataset's native train/test partition. No extra validation split is
invented here, matching the convention used for CIFAR-10 in this repo.

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so the dataset would otherwise be re-downloaded on every
fresh session. Instead we mount Google Drive and point `DATA_ROOT` at
a persistent Drive folder: `torchvision` only downloads into it once
and simply reads from it on every later run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/mnist

In [ ]:
from mnist_conv_autoencoder import get_mnist_dataloaders

BATCH_SIZE = 128

train_loader, test_loader = get_mnist_dataloaders(DATA_ROOT, BATCH_SIZE)
print(f"train batches = {len(train_loader)}, test batches = {len(test_loader)}")

## 5. Train

Explicit train/eval loops (`train_one_epoch`, `evaluate`), MSE
reconstruction loss (image vs. its own reconstruction, no labels used),
Adam optimizer.

In [ ]:
import torch.nn as nn
from mnist_conv_autoencoder import train_one_epoch, evaluate

N_EPOCHS = 10
LR = 1e-3

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_mse": [], "test_mse": []}

for epoch in range(N_EPOCHS):
    train_mse = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    test_mse = evaluate(model, test_loader, loss_fn, device)

    history["train_mse"].append(train_mse)
    history["test_mse"].append(test_mse)

    print(f"epoch {epoch + 1:3d}/{N_EPOCHS}  train_mse={train_mse:.4f}  test_mse={test_mse:.4f}")

## 6. Reconstruction-loss curve (quantitative result over training)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_mse"], label="train")
ax.plot(history["test_mse"], label="test")
ax.set_xlabel("epoch")
ax.set_ylabel("pixel-wise MSE")
ax.set_title("Reconstruction loss")
ax.legend()
fig.tight_layout()
plt.show()

print(f"final test_mse = {history['test_mse'][-1]:.4f}")

## 7. Qualitative + quantitative check: original vs. reconstruction

Per-project convention: always look at concrete examples, not only
scalar metrics. Show 5 real test images (top row) next to the model's
reconstruction (bottom row), with the per-image reconstruction MSE
(computed against that original image) printed under each
reconstruction -- the quantitative result shown with respect to the
original.

In [ ]:
from mnist_conv_autoencoder import visualize_reconstructions

sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

visualize_reconstructions(model, sample_images, sample_labels, num_examples=5)

## 8. Test-set average reconstruction MSE

A single aggregate number (over the full test split, not just the 5
plotted examples) for comparison across runs/configs.

In [ ]:
final_test_mse = evaluate(model, test_loader, loss_fn, device)
print(f"average test-set reconstruction MSE (latent_dim={LATENT_DIM}): {final_test_mse:.4f}")